In [0]:
import smtplib
import traceback
from datetime import datetime, timedelta
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from typing import List, Sequence

import pyspark.sql.functions as F
from pyspark.sql import types as T

In [0]:
# =========================
# Email
# =========================
SMTP_HOST = "mailrelay.eu.elcompanies.net"
SMTP_PORT = 25
# SMTP_USERNAME = "your_account@example.com"
# SMTP_PASSWORD = "your_smtp_password"
SMTP_USE_TLS = False

# sa-hk-kwn-apac-cdp@estee.com
FROM_ADDR = "sa-hk-kwn-apac-cdp@estee.com"

MAX_ROWS = 10
MAX_COLS = 8

TEXT_FALLBACK = "Please view this email in HTML format to see the table preview."

In [0]:
def build_html_table_from_spark_df(df, max_rows: int = MAX_ROWS, max_cols: int = MAX_COLS) -> str:
    """Build an HTML preview table from Spark DataFrame.

    Boundary:
    - only first `max_rows` rows
    - only first `max_cols` columns
    """
    selected_cols = list(df.columns[:max_cols])
    preview_pdf = df.select(*selected_cols).limit(max_rows).toPandas()

    row_count_text = "unknown"
    col_count_text = str(len(df.columns))
    try:
        row_count_text = str(df.count())
    except Exception:
        pass

    table_html = preview_pdf.to_html(
        index=False,
        border=1,
        justify='center',
        classes='data-table',
        na_rep='N/A'
    )

    # return f"""
    # <h3>Spark DataFrame Preview</h3>
    # <p>Total rows: {row_count_text}, total columns: {col_count_text}</p>
    # <p>Showing first {max_rows} rows and first {max_cols} columns only.</p>
    # {table_html}
    # """

    css_style = """
        <style>
            .data-table {
                border-collapse: collapse;  /* 合并边框为单线 */
                width: 70%;
                margin-left: 0;
                font-family: Arial, sans-serif;
                font-size: 12px;
            }
            .data-table th,
            .data-table td {
                border: 1px solid #ccc;  /* 单线边框 */
                padding: 6px;
                text-align: left;  /* 强制居中对齐 */
            }
            .data-table th {
                background-color: #f8f9fa;  /* 表头背景色 */
                border-bottom: 2px solid #666;  /* 加粗表头下边框 */
            }
        </style>
        """

    # 包裹HTML结构增强兼容性
    full_html = f"""
    <html>
    <head>{css_style}</head>
    <body>
        {table_html}
        <span>Total rows: {row_count_text}, total columns: {col_count_text}</span><br>
        <span>Showing first {max_rows} rows and first {max_cols} columns only.</span><br>
    </body>
    </html>
    """

    return full_html


In [0]:
def build_html_table_fragment(df, max_rows: int = MAX_ROWS, max_cols: int | None = MAX_COLS):
    """Build an HTML <table> fragment (no <html>/<body> wrapper) from a Spark DataFrame.

    Returns (html_fragment, row_count_text, col_count_text).
    Returns a placeholder paragraph if df is None.
    """
    if df is None:
        return "<p>No data available.</p>", "0", "0"

    selected_cols = list(df.columns) if max_cols is None else list(df.columns[:max_cols])
    pdf = df.select(*selected_cols).limit(max_rows).toPandas()

    col_count_text = str(len(df.columns))
    row_count_text = "unknown"
    try:
        row_count_text = str(df.count())
    except Exception:
        pass

    html = pdf.to_html(
        index=False,
        border=1,
        justify='center',
        classes='data-table',
        na_rep='N/A'
    )
    return html, row_count_text, col_count_text


In [0]:
def send_email(subject: str, html_body: str, to_addrs: Sequence[str], cc_addrs: Sequence[str] | None = None, bcc_addrs: Sequence[str] | None = None, custom_text: str = "") -> None:
    # bcc_addrs 密送名单
    cc_addrs = cc_addrs or []
    bcc_addrs = bcc_addrs or []

    all_recipients = list(to_addrs) + list(cc_addrs) + list(bcc_addrs)

    msg = MIMEMultipart("alternative")
    msg["Subject"] = subject
    msg["From"] = FROM_ADDR
    msg["To"] = ", ".join(to_addrs)
    if cc_addrs:
        msg["Cc"] = ", ".join(cc_addrs)

    plain_text = TEXT_FALLBACK if not custom_text else f"{custom_text}\n\n{TEXT_FALLBACK}"
    html_text = html_body if not custom_text else f"<p>{custom_text}</p>{html_body}"

    msg.attach(MIMEText(plain_text, "plain", "utf-8"))
    msg.attach(MIMEText(html_text, "html", "utf-8"))

    with smtplib.SMTP(SMTP_HOST, SMTP_PORT, timeout=30) as server:
        if SMTP_USE_TLS:
            server.starttls()
        # server.login(SMTP_USERNAME, SMTP_PASSWORD)
        server.sendmail(FROM_ADDR, all_recipients, msg.as_string())

In [0]:
# sample = [
#             ("A1001", "OK", 10),
#             ("A1002", "OK", 20),
#             ("A1003", "FAIL", 0),
#         ]
# df = spark.createDataFrame(sample, ["order_id", "status", "amount"])


# TO_ADDRS = [
#     "helv@estee.com",
#     "jezhan@estee.com",
# ]
# CC_ADDRS: List[str] = ["ege@estee.com"]
# BCC_ADDRS: List[str] = []

# SUBJECT = "[Simple Report] Test Preview By lvh"
# html_body = build_html_table_from_spark_df(df)

# send_email(
#     subject=SUBJECT,
#     html_body=html_body,
#     to_addrs=TO_ADDRS,
#     cc_addrs=CC_ADDRS,
#     bcc_addrs=BCC_ADDRS,
#     custom_text = "这是我发送的测试邮件"
# )